# Experiment Notebook: Hard-Slice Oversampling

Goal: test whether stronger hard-regime sampling improves overall performance versus standard domain-randomized tabular DRL.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append(str((Path.cwd().parent / "src").resolve()))
from ml_engine import research_upgrade_lab as lab

lab.set_seed(42)
df = lab.load_clean_dataset(Path.cwd())
weights = {"carbon": 0.35, "jct": 0.20, "tail": 0.25, "preempt": 0.10, "starvation": 0.10}
print("Dataset rows:", len(df))

In [ ]:
q_domain = lab.train_q_domain_randomized(df=df, episodes=180, reward_weights=weights, hard_bias=0.55)
q_hard = lab.train_q_domain_randomized(df=df, episodes=180, reward_weights=weights, hard_bias=0.85)

policy_tables = {
    "fcfs": None,
    "carbon": None,
    "srtf": None,
    "q_domain": q_domain,
    "q_hard": q_hard,
}

detail = lab.evaluate_policies(
    df=df,
    policy_tables=policy_tables,
    seeds=list(range(20)),
    noises=[15.0, 20.0],
    congestions=["moderate", "high"],
    capacities=[6, 8],
    reward_weights=weights,
)

scored = lab.compute_overall_score_robust(detail, weights)
hard = lab.summarize_hard_slice(scored)
display(hard.round(4))

fig, ax = plt.subplots(1, 1, figsize=(9, 4), dpi=110)
ax.bar(hard["policy"], hard["overall_score_mean"], yerr=hard["overall_score_ci95"], capsize=4)
ax.set_title("Hard Slice Overall Score (lower is better)")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()